# This notebook is part of the LinkedIn Learning Course: [Generative AI: Introduction to Diffusion Models for Text Generation](https://www.linkedin.com/learning/generative-ai-introduction-to-diffusion-models-for-text-generation/building-a-basic-text-diffusion-model-29722011?resume=false&u=35754684)

It is part of the `Section 3: Training a Text Diffusion Model`.




In [1]:
#Checking the Python version - for this colab notebook, I use 3.12.11
!python --version

Python 3.12.11


# Loading Modules

* `AutoTokenizer`
  * AutoTokenizer is a component within the Hugging Face transformers library that automatically selects and loads the appropriate tokenizer for a given pre-trained model. It streamlines the crucial first step of a Natural Language Processing (NLP) pipeline, which is to convert raw text into a numerical format that a machine learning model can process.

* `AutoModelForCausalLM` is a class
  * It is designed for causal language modeling, which means it's generating text one token at a time. The output will be a sequence of tokens that the model thinks is likely to come next.

* `DDPMScheduler`
  * The DDPMScheduler is an algorithm used in diffusion models to manage the iterative process of converting random noise into a coherent, high-quality image. In the context of the Hugging Face Diffusers library, it refers to the specific implementation based on the **Denoising Diffusion Probabilistic Models (DDPM)** paper.

In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from diffusers import DDPMScheduler
import torch
from torch import nn
import torch.nn.functional as F

from typing import List, Optional, Tuple, Union

## text_encoder.eval() mode below

In the context of a Python diffusion model using PyTorch, the line text_encoder.`eval()` switches the text_encoder model to evaluation mode. This is a crucial step when you are no longer training the model and are instead using it for inference, such as encoding a text prompt for image generation.


This command ensures that certain layers within the neural network behave correctly and deterministically for a final, consistent output.

Here’s a breakdown of what that means:

### Disables Dropout Layers:

During training, a model often includes dropout layers, which randomly "zero out" a fraction of the neurons in a layer to prevent overfitting. In evaluation mode, `text_encoder.eval()` ensures that these dropout layers are disabled. This is necessary to use all the neurons in the network and get a stable, consistent prediction every time.

### Affects Batch Normalization Layers:

While less common in a standard text encoder, models with Batch Normalization layers also behave differently in evaluation mode. During training, these layers calculate statistics (mean and standard deviation) from the current batch of data. In **evaluation mode**, they use the aggregated running statistics calculated over the entire training dataset instead.

### Works with torch.no_grad():

The `.eval()` call is frequently used in conjunction with with `torch.no_grad()`:.

The `.eval()` method controls layer behavior, while the `torch.no_grad()` context manager disables gradient computation, reducing memory consumption and speeding up the forward pass.

### Special Consideration - text to image

In a text-to-image diffusion pipeline, you would use `text_encoder.eval()` before encoding your text prompt, as you don't need to train the encoder further. This guarantees that the text embeddings you generate are stable and deterministic for the diffusion process.


In [3]:
# Initialize tokenizer and frozen text encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_encoder = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
text_encoder.eval()  # Freeze encoder for inference

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [4]:
# Diffusion noise scheduler with 1000 timesteps
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

In [5]:
def text_to_embedding(texts: List[str]) -> torch.Tensor:
    """Convert list of texts to BERT embeddings (last hidden layer).

    Args:
      text (List[str]): Lists of input text to be converted into an embedding.

    Returns:
      (torch.Tensor): the last element of this tuple, which is a single torch.Tensor
    """
    tokens = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = text_encoder(**tokens, output_hidden_states=True)
    return outputs.hidden_states[-1]  # Shape: [batch_size, seq_len, hidden_dim]

def add_noise(embedding: torch.Tensor, timesteps: List[int]) -> Tuple[torch.Tensor, torch.Tensor]:
    """Add noise to embeddings according to the
       Denoising Diffusion Probabilistic Model (DDPM) forward process.

    Args:
        embedding (torch.Tensor): The input tensor (e.g., text or image embedding).
        timestep (List[int]): The timestep for the forward diffusion process.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: A tuple containing the noisy embedding
        and the randomly generated noise tensor.

    """
    noise = torch.randn_like(embedding)
    noisy_embed = noise_scheduler.add_noise(embedding, noise, timesteps)
    return noisy_embed, noise

In [6]:
class SimpleDenoiser(nn.Module):
    """A minimal denoiser for a DDPM, consisting of a single linear layer.

    This class serves as a basic implementation of a denoiser model. In a real
    DDPM, this denoiser would be a more complex neural network, like a U-Net,
    capable of taking the timestep and conditional embedding into account.
    This version only processes the noisy input tensor.

    Args:
        hidden_dim (int): The dimension of the input and output tensors.
                          In a text-to-image pipeline, this would be the
                          embedding dimension.
    """
    def __init__(self, hidden_dim: int) -> None:
        """Initializes the SimpleDenoiser model."""
        super().__init__()
        self.linear = nn.Linear(hidden_dim, hidden_dim)

    def forward(self,
         x: torch.Tensor,
         t: Optional[torch.Tensor] = None,
         cond_emb: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Performs the forward pass to predict the noise.

        Although the forward method accepts `t` (timestep) and `cond_emb`
        (conditional embedding), this simple version does not use them,
        highlighting its minimalist nature.

        Args:
            x (torch.Tensor): The noisy input tensor (e.g., a noisy text embedding).
            t (Optional[torch.Tensor]): The timestep of the diffusion process.
                                        Not used in this simplified model.
            cond_emb (Optional[torch.Tensor]): The conditional embedding,
                                               such as a text embedding.
                                               Not used in this simplified model.

        Returns:
            (torch.Tensor): The predicted noise tensor, which has the same
                          shape and dimension as the input `x`.
        """
        return self.linear(x)

In [7]:
def train_denoiser_old(texts, epochs=5, batch_size=2, lr=1e-4):
    """Train the denoiser to predict noise given noisy embeddings."""
    denoiser = SimpleDenoiser(hidden_dim=768)
    optimizer = torch.optim.AdamW(denoiser.parameters(), lr=lr)

    for epoch in range(epochs):
        epoch_loss = 0
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            clean_embeds = text_to_embedding(batch_texts)

            # Sample random timesteps for each example in batch
            timesteps = torch.randint(0, 1000, (len(batch_texts),))

            # Add noise to embeddings
            noisy_embeds, true_noise = add_noise(clean_embeds, timesteps)

            # Predict noise with denoiser (only pass noisy_embeds)
            predicted_noise = denoiser(noisy_embeds)

            # Compute MSE loss
            loss = F.mse_loss(predicted_noise, true_noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss / (len(texts)/batch_size):.4f}")
    return denoiser

---

## Background on the train_noiser code


### `torch.randint(0, 1000, (len(batch_texts),))`

creates a 1D PyTorch tensor of a specific length, filled with random integers.

Here is a breakdown of the function's arguments:

* `torch.randint`:

This is the PyTorch function for generating a tensor with random integer values.

* `0:`

The lowest integer to be drawn from the distribution, inclusive.

* `1000:` The highest integer to be drawn from the distribution, exclusive. The random numbers will be between 0 and 999.

* `(len(batch_texts),)`:

This tuple specifies the shape of the output tensor.
`len(batch_texts)` calculates the number of items in the batch_texts list or array. This is likely the batch size in a machine learning context.
The trailing comma `(...,) `is necessary to define a tuple with a single element.




In [8]:
def train_denoiser(
    texts: List[str],
    epochs: int = 5,
    batch_size: int = 2,
    lr: float = 1e-4
) -> SimpleDenoiser:
    """Trains a SimpleDenoiser model to predict the noise added to text embeddings.

    This function implements a basic training loop for a Denoising Diffusion
    Probabilistic Model (DDPM). For each batch of texts, it performs the
    following steps:
    1. Converts the clean text into a numerical embedding.
    2. Samples a random timestep for each embedding.
    3. Adds noise to the embeddings according to the sampled timesteps.
    4. Feeds the noisy embeddings to the denoiser to predict the added noise.
    5. Calculates the Mean Squared Error (MSE) loss between the predicted and
       actual noise.
    6. Updates the denoiser's parameters via backpropagation and an optimizer.

    Args:
        texts (str): A list of text strings to use for training.
        epochs (int): The number of training epochs to run.
        batch_size (int): The number of text examples per training batch.
        lr (float): The learning rate for the AdamW optimizer.

    Returns:
        (SimpleDenoiser): The trained denoiser model instance.
    """
    denoiser = SimpleDenoiser(hidden_dim=768)
    optimizer = torch.optim.AdamW(denoiser.parameters(), lr=lr)

    for epoch in range(epochs):
        epoch_loss = 0
        for i in range(0, len(texts), batch_size): #range(start, stop, step)
            batch_texts = texts[i:i+batch_size]
            clean_embeds = text_to_embedding(batch_texts)

            # Sample random timesteps for each example in batch
            timesteps = torch.randint(0, 1000, (len(batch_texts),))

            # Add noise to embeddings (forward diffusion step)
            noisy_embeds, true_noise = add_noise(clean_embeds, timesteps)

            # Predict noise with denoiser
            predicted_noise = denoiser(noisy_embeds)

            # Compute Mean Squared Error (MSE) loss
            loss = F.mse_loss(predicted_noise, true_noise)

            # Backpropagation and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss / (len(texts)/batch_size):.4f}")
    return denoiser

## Background on some of the generate_text() code


### for t in reversed(range(steps))

means that the loop will iterate backward over a sequence of numbers.

```
steps = 5
for t in reversed(range(steps)):
    print(t)

Output:
4
3
2
1
0

```

### The command `noisy_embed = torch.randn_like(clean_embed)`

creates a new PyTorch tensor named `noisy_embed` with the exact same size, data type, and device as the existing tensor `clean_embed,` but it fills the new tensor with random numbers`.


### `torch.full((1,), t, dtype=torch.long)`

command creates a PyTorch tensor with a specific size, filled with a constant value and a defined data type.

Here is a breakdown of the components:

* `torch.full()`:

  The function used to create the tensor.

* `(1,)`:

  This is a tuple that defines the size (or shape) of the output tensor. `(1,)`: specifies a one-dimensional tensor with one element.

* `t`:

  This is the value used to fill the tensor. The tensor's single element will have the value of the variable t.

`dtype=torch.long`:

  This sets the data type of the tensor to a 64-bit signed integer (torch.int64), which is specified as torch.long for convenience.

### noisy_embed = noise_scheduler.step(pred_noise, t, noisy_embed).prev_sample

Putting it all together, the line of code describes a single iteration of the reverse (denoising) process in a diffusion model.

A pre-trained neural network predicts the noise (pred_noise) within the current noisy latent representation (noisy_embed).

The noise_scheduler uses this prediction, along with the current timestep t, to calculate a new, slightly denoised latent representation.

The result of this calculation (.prev_sample) replaces the previous noisy latent representation, preparing for the next step in the loop.


The line of code `noisy_embed = noise_scheduler.step(pred_noise, t, noisy_embed)`.

`prev_sample` performs a single denoising step during the image generation process of a diffusion model.


Here is a breakdown of what each component means in the context of a library like **Hugging Face's diffusers**:


#### noise_scheduler

This is a class that implements the specific mathematical method for updating a noisy sample based on the model's noise prediction. Different schedulers use different algorithms (e.g., DDPM, DDIM, Euler) and vary how noise is added or removed at each timestep.


#### noise_scheduler.step()

This method takes the current prediction from the diffusion model and uses it to compute a less noisy version of the input.

* `pred_noise`:

  This is the output of the neural network (e.g., a UNet) that was trained to predict the noise component of the noisy image.

* `t`:

  This is the current timestep in the denoising process. Diffusion models generate images by starting with pure noise and iteratively removing noise over a series of steps (timesteps). A typical process might have 1000 timesteps, going from a high value (very noisy) down to 0 (the final image).

* `noisy_embed`:

  This is the current noisy latent representation (or "embedding") that needs to be denoised. It serves as the input for the denoising process at this specific timestep.

#### .prev_sample

The `step()` method typically returns an object containing multiple values. The `.prev_sample` attribute explicitly retrieves the computed output for the previous timestep, which is the less noisy version of `noisy_embed`.

#### noisy_embed = ...

The result `(prev_sample)` is then assigned back to the `noisy_embed` variable. In the next iteration of the denoising loop, this new, slightly less noisy noisy_embed will be fed back into the model along with the next timestep (t-1) to continue the denoising process.


In [9]:
def generate_text(prompt:str,
                  denoiser: SimpleDenoiser,
                  steps: int=50
  ) -> List[str]:
  """This function Generates text using a denoiser model.

    Args:
        promt (str): A text strings to use for denoising.
        denoiser (SimpleDenoiser): TThe trained denoiser model instance.
        steps (int): The number of steps


    Returns:
        (List[str]): The List of denoised strings.

  """
  clean_embed = text_to_embedding([prompt])
  noisy_embed = torch.randn_like(clean_embed)

  for t in reversed(range(steps)):
    timestep = torch.full((1,),t, dtype=torch.long)
    with torch.no_grad():
      pred_noise = denoiser(noisy_embed)
      noisy_embed = noise_scheduler.step(pred_noise, t, noisy_embed).prev_sample

  with torch.no_grad():
    # Pass the embedded inputs to the 'bert' attribute
    #outputs = text_encoder.bert(inputs_embeds=noisy_embed)
    # Access the logits from the 'cls' attribute (MLM head)
    logits = text_encoder(inputs_embeds=noisy_embed).logits
    predicted_ids = logits.argmax(dim=-1)
    generated_text = tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
    return generated_text

In [10]:
sample_texts = [
    "The cat sat on the mat",
    "Diffusion models are powerful",
    "Natural Language Processing is evolving",
    "Machine learning requires large datasets",
    "Attention mechanism revolutionalized NLP"
]

In [11]:
trained_denoised = train_denoiser(sample_texts, epochs=5)

Epoch 1/5 - Loss: 1.5940
Epoch 2/5 - Loss: 1.5016
Epoch 3/5 - Loss: 1.4709
Epoch 4/5 - Loss: 1.5684
Epoch 5/5 - Loss: 1.5514


## Testing the Trained dataset below

Notice the sample_texts list is very sparse meaning the training set was very small. As a result, it is highly unlikely that the test below will be successful.

In fact, as you will see the results are not very close. To get better, I would expect to use a much lager training set.

The point of this excercise is really to show that results, though not correct, are much better than the previous notebook that did not take advantage of training.


In [12]:
output = generate_text("The future of AI is", trained_denoised)
output

['family in who many toll family family']